# Fase 2e: Analisis de postprocesado CT

Evalua umbral y filtrado de componentes conectadas sobre los modelos CT ya entrenados. No reentrena modelos.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib_cache'))

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import config
from src.data.segmentation import build_ct_segmentation_dataframe, split_segmentation_dataframe, SegmentationDataset, SegmentationPairTransform
from src.evaluation.segmentation_postprocessing import binary_mask_metrics, postprocess_binary_mask
from src.models.segmentation import build_segmentation_model

config.NUM_WORKERS = 0
pd.set_option('display.max_columns', 50)

In [ ]:
ct_seg_df = build_ct_segmentation_dataframe(
    config.CT_DIR,
    config.CT_DIR / 'processed_segmentation_slices',
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=True,
    overwrite=False,
)
_, _, test_df = split_segmentation_dataframe(ct_seg_df, random_seed=config.RANDOM_SEED, group_col='study_id')
test_loader = DataLoader(
    SegmentationDataset(test_df, transform=SegmentationPairTransform(config.CT_IMAGE_SIZE, 1, is_train=False), in_channels=1),
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

In [ ]:
def collect_probs(experiment, architecture='attention_unet', base_features=32):
    checkpoint = torch.load(config.MODELS_DIR / 'segmentation' / 'ct' / f'{experiment}_full.pt', map_location='cpu')
    model = build_segmentation_model(architecture, in_channels=1, base_features=base_features)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    probs = []
    targets = []
    with torch.no_grad():
        for images, masks in test_loader:
            probs.append(torch.sigmoid(model(images)).cpu().numpy()[:, 0])
            targets.append(masks.cpu().numpy()[:, 0].astype(bool))
    return np.concatenate(probs), np.concatenate(targets)

def evaluate_postprocessing(probs, targets, threshold, min_component_area=0, close_kernel_size=0):
    predictions = probs >= threshold
    processed = np.zeros_like(predictions, dtype=bool)
    for idx in range(predictions.shape[0]):
        processed[idx] = postprocess_binary_mask(
            predictions[idx],
            min_component_area=min_component_area,
            keep_largest_component=False,
            close_kernel_size=close_kernel_size,
        )
    return binary_mask_metrics(processed, targets)

In [ ]:
probs, targets = collect_probs('ct_attention_unet_segmentation', base_features=32)
rows = []
for threshold in [0.35, 0.4, 0.45, 0.5, 0.55, 0.6]:
    for min_area in [0, 5, 10, 20, 40]:
        metrics = evaluate_postprocessing(probs, targets, threshold, min_component_area=min_area)
        rows.append({
            'threshold': threshold,
            'min_component_area': min_area,
            **metrics,
        })
post_df = pd.DataFrame(rows).sort_values('dice', ascending=False)
display(post_df.head(10))